In [0]:
bronze_path = "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

dept_schema = StructType([ 
  StructField("dept", StringType(), True), 
  StructField("location", StringType(), True), 
  StructField("manager", StringType(), True), 
  StructField("budget", IntegerType(), True), 
])

df_dept = spark.read.option("multiLine", "true").json('/Volumes/hr_project_catalog/hr_schema/hr_project_volume/raw/departments.json',schema = dept_schema)

df_dept.write.mode("overwrite").parquet(bronze_path + "departments")

display(df_dept)

dept,location,manager,budget
Engineering,New York,Sara,500000
Marketing,Chicago,Tom,300000
HR,Austin,Asha,200000
Finance,Dallas,Leo,400000


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

emp_schema = StructType([ 
  StructField("id", IntegerType(), False), 
  StructField("name", StringType(), True), 
  StructField("dept", StringType(), True), 
  StructField("salary", IntegerType(), True), 
  StructField("join_date", StringType(), True), 
  StructField("gender", StringType(), True), 
  StructField("age", IntegerType(), True), 
  StructField("is_active", BooleanType(), True), 
  StructField("bonus", IntegerType(), True), 
  StructField("rating", IntegerType(), True), 
]) 

df_emp = spark.read.csv(
    '/Volumes/hr_project_catalog/hr_schema/hr_project_volume/raw/employees.csv',
    header=True,
    schema=emp_schema
)

df_emp.write.mode("overwrite").parquet(bronze_path + "employees")

display(df_emp)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5
4,David,HR,61000,2022-01-10,M,27,true,6100,4
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3


In [0]:
from pyspark.sql.functions import col

df_hires = spark.read.parquet(
    '/Volumes/hr_project_catalog/hr_schema/hr_project_volume/raw/new_hires.parquet'
)

df_hires = df_hires.select(
    col("id").cast("int"),
    "name",
    "dept",
    col("salary").cast("int"),
    "join_date",
    "gender",
    col("age").cast("int"),
    "is_active",
    col("bonus").cast("int"),
    col("rating").cast("int")
)

df_hires.write.mode("overwrite").parquet(bronze_path + "new_hires")

display(df_hires)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

bands_schema = StructType([ 
    StructField("dept", StringType(), True), 
    StructField("gender", StringType(), True), 
    StructField("band_label", StringType(), True)
]) 

df_bands = spark.read.csv('/Volumes/hr_project_catalog/hr_schema/hr_project_volume/raw/salary_bands.csv', header=True, schema=bands_schema)

df_bands.write.mode("overwrite").parquet(bronze_path + "salary_bands")

display(df_bands)

dept,gender,band_label
Engineering,F,Band-A
Engineering,M,Band-A
Marketing,F,Band-B
Marketing,M,Band-B
HR,F,Band-C
HR,M,Band-C
Finance,F,Band-B
Finance,M,Band-B


In [0]:
# Employees
df_emp.printSchema()

# Departments
df_dept.printSchema()

# New Hires
df_hires.printSchema()

# Salary Bands
df_bands.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- join_date: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- bonus: integer (nullable = true)
 |-- rating: integer (nullable = true)

root
 |-- dept: string (nullable = true)
 |-- location: string (nullable = true)
 |-- manager: string (nullable = true)
 |-- budget: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- join_date: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- bonus: integer (nullable = true)
 |-- rating: integer (nullable = true)

root
 |-- dept: string (nullable = true)
 |-- gender: string (nullab

In [0]:
print("Employees columns:", df_emp.columns)
print("New hires columns:", df_hires.columns)
print("Departments columns:", df_dept.columns)
print("Salary bands columns:", df_bands.columns)

Employees columns: ['id', 'name', 'dept', 'salary', 'join_date', 'gender', 'age', 'is_active', 'bonus', 'rating']
New hires columns: ['id', 'name', 'dept', 'salary', 'join_date', 'gender', 'age', 'is_active', 'bonus', 'rating']
Departments columns: ['dept', 'location', 'manager', 'budget']
Salary bands columns: ['dept', 'gender', 'band_label']


In [0]:
# Compare column names
print("Columns match:", df_emp.columns == df_hires.columns)

# Compare schema (data types)
print("Schema match:", df_emp.schema == df_hires.schema)

Columns match: True
Schema match: True


In [0]:
print("Employees dtypes:", df_emp.dtypes)
print("New hires dtypes:", df_hires.dtypes)

Employees dtypes: [('id', 'int'), ('name', 'string'), ('dept', 'string'), ('salary', 'int'), ('join_date', 'string'), ('gender', 'string'), ('age', 'int'), ('is_active', 'boolean'), ('bonus', 'int'), ('rating', 'int')]
New hires dtypes: [('id', 'int'), ('name', 'string'), ('dept', 'string'), ('salary', 'int'), ('join_date', 'string'), ('gender', 'string'), ('age', 'int'), ('is_active', 'boolean'), ('bonus', 'int'), ('rating', 'int')]
